<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.1/1B_agent_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Fundamentos de Agentes de IA

## Objetivos de Aprendizaje
- Comprender qué es un agente de IA y cuáles son sus componentes clave.
- Entender el ciclo de razonamiento (ReAct) que sigue un agente para resolver problemas.
- Implementar un agente simple desde cero en Python (Vanilla Code).
- Reconocer las limitaciones de un agente básico y la necesidad de frameworks como LangChain o CrewAI.

## ¿Qué es un Agente de IA?

Un agente de IA no es simplemente un modelo de lenguaje (LLM) que responde a preguntas. Es un sistema más avanzado que utiliza un LLM como su **cerebro (core engine)** para razonar y tomar decisiones. A diferencia de una simple llamada a una API, un agente puede:

1.  **Descomponer un objetivo complejo** en una secuencia de pasos intermedios.
2.  **Interactuar con herramientas externas** (APIs, bases de datos, funciones de código) para obtener información o ejecutar acciones en el mundo real.
3.  **Observar los resultados** de esas acciones y ajustar su plan en consecuencia.
4.  **Repetir este ciclo** hasta que el objetivo original se haya cumplido.

Piénsalo como un becario inteligente: le das una tarea de alto nivel (ej. "Investiga el precio de las acciones de Apple y dime si es un buen momento para comprar"), y él solo descubre qué herramientas usar (búsqueda web, una API financiera), cómo usarlas y cómo interpretar los resultados para darte una recomendación.

### Componentes Clave de un Agente

Un agente, en su forma más básica, se compone de tres elementos principales:

1.  **Cerebro (Core Engine)**: El LLM que impulsa al agente. Es responsable del razonamiento, la planificación y la toma de decisiones.
2.  **Memoria (Memory)**: Un sistema para almacenar y recuperar información de la conversación actual (memoria a corto plazo) o de interacciones pasadas (memoria a largo plazo). Esto le da contexto al agente.
3.  **Herramientas (Tools)**: Funciones o APIs que el agente puede "llamar" para interactuar con el mundo exterior. Esto supera la limitación del conocimiento estático del LLM.

## Implementación de un Agente Básico en Python

In [14]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q openai python-dotenv


In [15]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
import re
import json
from openai import OpenAI
from datetime import datetime

# --- 1. Configuración del Cliente OpenAI ---
# Asegúrate de tener las variables de entorno LLM_BASE_URL y LLM_API_KEY configuradas
try:
    client = OpenAI(
        base_url=os.environ.get("LLM_BASE_URL"),
        api_key=os.environ.get("LLM_API_KEY")
    )
    print("✅ Cliente OpenAI configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando el cliente: {e}")
    client = None

✅ Cliente OpenAI configurado correctamente.


### 2. Definición de las Herramientas (Tools)

Vamos a crear dos herramientas muy simples que nuestro agente podrá usar:

In [16]:
def get_current_time(args):
    """Devuelve la fecha y hora actual."""
    return f"La fecha y hora actual es: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def search_web(args):
    """Simula una búsqueda web para un término dado."""
    query = args.get("query", "")
    # En una implementación real, aquí llamaríamos a una API de búsqueda (ej. Google, Bing)
    print(f"🔎 Buscando en la web: '{query}'...")
    if "elon musk" in query.lower():
        return "Elon Musk es el CEO de SpaceX y Tesla."
    elif "inteligencia artificial" in query.lower():
        return "La IA es un campo de la informática dedicado a crear sistemas que pueden realizar tareas que normalmente requieren inteligencia humana."
    else:
        return f"No se encontraron resultados para '{query}'."

# Mapeo de herramientas para que el agente sepa qué funciones puede llamar
tools = {
    "get_current_time": {
        "function": get_current_time,
        "description": "Útil para obtener la fecha y hora actual.",
        "args": {}
    },
    "search_web": {
        "function": search_web,
        "description": "Útil para buscar información en internet sobre personas, lugares o conceptos.",
        "args": {"query": "la pregunta a buscar"}
    }
}

print("✅ Herramientas del agente definidas.")

✅ Herramientas del agente definidas.


In [17]:
# Definimos un decodificador JSON
# porque Una expresión regular no entiende conceptualmente que unas llaves están anidadas dentro de otras.
def extract_action(text):
    if "Action:" not in text:
        return None

    action_part = text.split("Action:", 1)[1].strip()

    decoder = json.JSONDecoder()

    try:
        action_json, _ = decoder.raw_decode(action_part)
        return action_json
    except json.JSONDecodeError:
        return None

### 3. El Cerebro del Agente y el Ciclo ReAct

Ahora, la parte más importante: el **ciclo de razonamiento**. Usaremos un enfoque llamado **ReAct (Reason + Act)**. En cada paso, el LLM decide una de estas tres cosas:

1.  **Reason (Razonar)**: Piensa cuál es el siguiente paso lógico para alcanzar el objetivo.
2.  **Act (Actuar)**: Elige y utiliza una de las herramientas disponibles.
3.  **Answer (Responder)**: Si ya tiene suficiente información, da la respuesta final al usuario.

Para guiar al LLM, usaremos un **prompt de sistema** muy específico que le enseñe este patrón de pensamiento.

In [20]:


# Este prompt es deliberadamente más detallado que una instrucción tradicional.
# En un agente basado en ReAct, el LLM no solo debe responder al usuario:
# también debe comunicarse con el código que interpreta sus decisiones y ejecuta
# las herramientas. Por eso es necesario definir con precisión el formato esperado.
#
# Las reglas adicionales reducen ambigüedades en la salida del modelo y evitan
# comportamientos como:
# - inventar resultados de herramientas;
# - escribir la Observation antes de que la herramienta sea ejecutada;
# - generar JSON incompleto o con formatos distintos a los esperados;
# - repetir acciones ya realizadas;
# - entregar una respuesta final antes de resolver todas las partes de la consulta.
#
# En este caso, el prompt funciona como un "contrato de comunicación" entre
# el LLM y el orquestador Python. Mientras más estructurada debe ser la salida
# del modelo para ser procesada automáticamente, más explícitas deben ser las
# instrucciones que definen ese protocolo.

def create_system_prompt(tools):
    tool_descs = []

    for name, details in tools.items():
        tool_descs.append(
            f"- {name}: {details['description']} "
            f"Argumentos: {details['args']}"
        )

    tools_text = "\n".join(tool_descs)

    prompt = f"""
Eres un agente de IA capaz de utilizar herramientas para resolver solicitudes del usuario.

Herramientas disponibles:
{tools_text}

Debes trabajar siguiendo un ciclo tipo ReAct:
Reason -> Action -> Observation -> Reason -> ... -> Final Answer

En cada turno debes producir SOLO una de estas dos estructuras:

1. Si necesitas utilizar una herramienta:

Reason: <explica brevemente qué información necesitas obtener o qué acción debes realizar>
Action: {{"tool": "<nombre_de_la_herramienta>", "args": {{...}}}}

2. Si ya tienes toda la información necesaria para responder:

Final Answer: <respuesta final para el usuario>

IMPORTANTE:
- "Reason:" debe ser breve, claro y orientado a justificar la siguiente acción.
- Toda llamada a herramienta DEBE comenzar exactamente con "Action:".
- El contenido de "Action:" debe ser un único objeto JSON válido.
- El JSON debe contener siempre las claves "tool" y "args".
- "tool" debe corresponder exactamente al nombre de una herramienta disponible.
- "args" debe ser un objeto JSON, aunque la herramienta no requiera argumentos.
- Nunca respondas únicamente con un bloque JSON.
- No uses bloques Markdown ni ``` alrededor del JSON.
- Nunca escribas "Observation:"; esa parte será generada por el sistema después de ejecutar la herramienta.
- Nunca inventes el resultado de una herramienta.
- Después del JSON de "Action:" no escribas ninguna explicación adicional.
- No repitas una herramienta si la Observation indica que ya se ejecutó correctamente, salvo que sea estrictamente necesario con argumentos diferentes.
- Debes resolver TODAS las partes de la solicitud antes de entregar "Final Answer:".
- Si todavía falta información para responder completamente, debes utilizar otra herramienta.
- Cuando ya tengas toda la información necesaria, responde únicamente con "Final Answer:" seguido de la respuesta.
- No inventes datos que no hayan sido entregados por el usuario o por una Observation.
"""

    return prompt

def run_agent(user_query, client, tools):
    if not client:
        print("❌ Cliente no inicializado.")
        return

    system_prompt = create_system_prompt(tools)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]

    print(f"--- Agente iniciado para la consulta: '{user_query}' ---")

    for _ in range(5): # Limitar a 5 iteraciones para evitar bucles infinitos
        response = client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "mistral-small-latest"),
            messages=messages,
            temperature=0,
            max_tokens=500
        )

        print("\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - -\n")
        text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(f"🤖 Pensamiento del Agente: [{text}]")

        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            print(f"--- ✅ Agente ha finalizado --- ")
            return final_answer

        action_json = extract_action(text)

        if action_json:
            try:
                tool_name = action_json["tool"]
                tool_args = action_json.get("args", {})

                if tool_name in tools:
                    observation = tools[tool_name]["function"](tool_args)

                    print(f"🔧 Ejecutando herramienta: {tool_name}")
                    print(f"👁️ Observación real: {observation}")

                    messages.append({
                        "role": "user",
                        "content": f"Observation: {observation}"
                    })

                else:
                    messages.append({
                        "role": "user",
                        "content": f"Observation: Herramienta '{tool_name}' desconocida."
                    })

            except Exception as e:
                messages.append({
                    "role": "user",
                    "content": f"Observation: Error al ejecutar la acción - {str(e)}"
                })

        else:
            print("--- ⚠️ El agente no pudo determinar una acción y se detuvo. ---")
            return text

    print("--- 🛑 Límite de iteraciones alcanzado. ---")
    return "El agente no pudo completar la tarea en el número máximo de pasos."

print("✅ Lógica del agente definida.")

✅ Lógica del agente definida.


### 4. Ejecución del Agente

Ahora, pongamos a nuestro agente a trabajar con una pregunta que requiere usar una herramienta.

In [21]:
import json
final_response = run_agent("¿Quién es Elon Musk y qué hora es?", client, tools)
print(f"\n🏁 Respuesta Final del Agente: {final_response}")

--- Agente iniciado para la consulta: '¿Quién es Elon Musk y qué hora es?' ---

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

🤖 Pensamiento del Agente: [Reason: Necesito obtener información sobre quién es Elon Musk y también la hora actual para responder a la solicitud del usuario.
Action: {"tool": "search_web", "args": {"query": "¿Quién es Elon Musk?"}}]
🔎 Buscando en la web: '¿Quién es Elon Musk?'...
🔧 Ejecutando herramienta: search_web
👁️ Observación real: Elon Musk es el CEO de SpaceX y Tesla.

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

🤖 Pensamiento del Agente: [Reason: Necesito obtener la fecha y hora actual para completar la respuesta.
Action: {"tool": "get_current_time", "args": {}}]
🔧 Ejecutando herramienta: get_current_time
👁️ Observación real: La fecha y hora actual es: 2026-09-30 23:04:37

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

🤖 Pensamiento del Agente: [Final Answer:
Elon Musk es un empresario, ingeniero e inventor sud

## Conclusiones y Próximos Pasos

Hemos construido un agente funcional desde cero. Sin embargo, hemos tenido que manejar mucha lógica compleja:

- **Análisis de la respuesta del LLM**: Usar expresiones regulares (`re`) y `json.loads` para extraer la acción es frágil y propenso a errores.
- **Gestión del prompt**: Construir y actualizar el prompt manualmente es tedioso.
- **Manejo del ciclo**: El bucle `for` con la lógica de parada es repetitivo.
- **Escalabilidad**: Añadir más herramientas, gestionar la memoria o implementar planes más complejos se volvería muy difícil.

**Aquí es donde entran los frameworks como LangChain y CrewAI.**

Estos frameworks abstraen toda esta complejidad, permitiéndonos definir agentes, herramientas y tareas de una manera mucho más declarativa y robusta. En los próximos notebooks, veremos cómo recrear este mismo agente usando estas herramientas para apreciar la diferencia en simplicidad y potencia.